|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>The block allocator<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: build the allocator and the page table<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

Build the allocator.

You need no tensors and no GPU. This is stage 06 of the ladder. It is pure
bookkeeping. Get it correct on its own, before a kernel must read through
it.

In [ ]:
### run this cell

BLOCK    = 16                # tokens per block
N_BLOCKS = 4096              # blocks in the pool
lengths  = rng.lognormal(mean=np.log(120), sigma=0.9, size=2000).astype(int) + 1

print(f'pool holds {N_BLOCKS*BLOCK:,} tokens in {N_BLOCKS:,} blocks of {BLOCK}')

# Exercise 1: the free list

You need a pool of blocks, a stack of the free ones, and one reference count
per block. The reference count looks unnecessary now. Exercise 4 explains
it.

In [ ]:
class BlockAllocator:
  def __init__(self, n_blocks):
    self.free = 
    self.ref  = [0] * n_blocks      # how many sequences point at each block

  def allocate(self):
    # hand out a free block, or say so clearly if there are none
    

  def release(self, b):
    # One owner fewer. Free the block only when no owner remains.
    

  def share(self, b):
    # a second sequence starts pointing at this block
    

  @property
  def used(self):
    return len(self.ref) - len(self.free)

a = BlockAllocator(8)
x, y = a.allocate(), a.allocate()
print('used after 2 allocations:', a.used)
a.release(x)
print('used after 1 release:    ', a.used)

# Exercise 2: the block table

Each sequence holds one table. The table maps a logical token position onto a
physical slot in the pool. It grows one block at a time, as the sequence
grows.

In [ ]:
class BlockTable:
  """One sequence's view of the pool."""
  def __init__(self, allocator, block_size):
    self.alloc  = allocator
    self.bs     = block_size
    self.blocks = []      # logical block index -> physical block id
    self.n      = 0       # tokens held

  def append_token(self):
    # only ask for a new block when the current one is full
    
    self.n += 1

  def slot_index(self, pos):
    """logical token position -> flat slot in the pool.
    This is the 'slot mapping' you will see all over serving code."""
    return 

  def free(self):
    

alloc = BlockAllocator(N_BLOCKS)
t = BlockTable(alloc, BLOCK)
for _ in range(40):
  t.append_token()
print(f'40 tokens -> {len(t.blocks)} blocks: {t.blocks}')
print(f'position 0  -> slot {t.slot_index(0)}')
print(f'position 17 -> slot {t.slot_index(17)}')

# Exercise 3: how many sequences fit now?

Admit requests from the workload until the allocator refuses. Then compare
your answer against a reservation of `max_len` for each request.

In [ ]:
alloc  = BlockAllocator(N_BLOCKS)
tables = []
admitted = 0

# admit sequences until the allocator refuses
for L in lengths:
  t = BlockTable(alloc, BLOCK)
  try:
    
  except MemoryError:
    t.free()
    break
  tables.append(t); admitted += 1

held  = 
used  = 
print(f'admitted {admitted} sequences before the pool ran out')
print(f'tokens held {held:,}, tokens used {used:,}  -> {100*(1-used/held):.1f}% wasted')

MAX_LEN = 2048
contig  = (N_BLOCKS*BLOCK) // MAX_LEN
print(f'\ncontiguous, reserving {MAX_LEN}: {contig} sequences')
print(f'paged:                     {admitted} sequences   ({admitted/contig:.0f}x)')

# Exercise 4: two sequences, one prompt

Parallel sampling asks the model for four replies to one prompt. All four
replies share the same K and V for the prompt.

A page table makes that free.

In [ ]:
alloc = BlockAllocator(N_BLOCKS)
before = alloc.used

parent = BlockTable(alloc, BLOCK)
for _ in range(64):
  parent.append_token()
after_parent = alloc.used

# four samples from the same prompt. They all start from the same tokens,
# so they can point at the same physical blocks. Use share(), not allocate().
children = []
for _ in range(4):
  c = BlockTable(alloc, BLOCK)
  c.blocks = 
  c.n = parent.n
  children.append(c)

print(f'prompt of 64 tokens costs      {after_parent - before} blocks')
print(f'4 samples sharing it cost      {alloc.used - after_parent} more')
print(f'4 samples copying it would be  {4*(after_parent-before)} more')

for c in children:
  c.free()
print(f'\nafter the children leave, still held: {alloc.used} blocks (the parent)')

### Before you open the solution

1. Your waste in Exercise 3 is a few percent, not ninety. What is the hard
   upper limit on wasted tokens per sequence? What sets that limit?
2. In Exercise 4 four samples shared one prompt for almost nothing. Can one
   contiguous buffer per sequence do that?
3. One of those four children now makes a token that the others do not make.
   What must happen? Which block is the problem? What is the cheapest correct
   fix?